In [ ]:
# Fixed version of your code that handles MoreComments properly

# First, let's see what 'news' contains and set up proper Reddit connection
# Assuming you have a Reddit instance and subreddit already set up

# Method 1: Simple fix for your existing code
def print_posts_and_comments_simple(posts):
    """Simple method to print posts and comments, handling MoreComments"""
    for n in posts:
        print(f"Title: {n.title}")
        print(f"Body: {n.selftext}")
        print("Comments:")
        
        # Replace MoreComments objects with actual comments
        n.comments.replace_more(limit=0)  # This removes MoreComments placeholders
        
        for c in n.comments.list():  # Use .list() to get all comments
            if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']:
                print(f'    {c.body}')
        print("-" * 50)

# Method 2: More robust version with error handling
def print_posts_and_comments_robust(posts):
    """Robust method with better error handling"""
    for i, n in enumerate(posts, 1):
        print(f"\n--- Post {i} ---")
        print(f"Title: {n.title}")
        print(f"Body: {n.selftext[:200]}{'...' if len(n.selftext) > 200 else ''}")
        print(f"Score: {n.score}, Comments: {n.num_comments}")
        
        try:
            # Replace MoreComments with actual comments
            n.comments.replace_more(limit=0)
            
            comment_count = 0
            for c in n.comments.list():
                if comment_count >= 5:  # Limit to first 5 comments
                    break
                    
                if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']:
                    print(f"  Comment {comment_count + 1}: {c.body[:100]}{'...' if len(c.body) > 100 else ''}")
                    comment_count += 1
                    
        except Exception as e:
            print(f"  Error loading comments: {e}")
        
        print("-" * 50)

# Method 3: Get comments as a list (similar to your original function)
def get_posts_with_comments_list(posts, max_comments_per_post=10):
    """Convert posts to the format you originally wanted"""
    result = []
    
    for n in posts:
        # Extract post data
        post_data = {
            "title": n.title,
            "body": n.selftext if n.selftext else ""
        }
        
        # Extract comments
        comments = []
        try:
            n.comments.replace_more(limit=0)
            
            comment_count = 0
            for c in n.comments.list():
                if comment_count >= max_comments_per_post:
                    break
                    
                if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']:
                    comments.append(c.body)
                    comment_count += 1
                    
        except Exception as e:
            print(f"Error loading comments for post '{n.title}': {e}")
            comments = []
        
        # Format as requested
        formatted_post = {
            "post": post_data,
            "comments": comments
        }
        
        result.append(formatted_post)
    
    return result

# Example usage:
# If you have your 'news' variable, you can now use:
# print_posts_and_comments_simple(news)
# or
# posts_data = get_posts_with_comments_list(news, max_comments_per_post=5)
# print(json.dumps(posts_data, indent=2))

print("Fixed Reddit comment handling functions loaded!")
print("Use print_posts_and_comments_simple(news) to fix your current code")
print("Or use get_posts_with_comments_list(news) to get the JSON format you wanted")


In [ ]:
# Quick fix for your current error - run this cell to fix your existing code

# Replace your problematic code with this:
for n in news:
    print(f"Title: {n.title}")
    print(f"Body: {n.selftext}")
    print("Comments:")
    
    # This is the key fix - replace MoreComments objects with actual comments
    n.comments.replace_more(limit=0)
    
    # Now iterate through the actual comments
    for c in n.comments.list():
        if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']:
            print(f'    {c.body}')
    print("-" * 50)


In [ ]:
# Function to iterate through the entire comment forest (all nested comments)

def iterate_comment_forest(comments, level=0, max_depth=None):
    """
    Recursively iterate through all comments in a comment forest
    
    Args:
        comments: Reddit comment object or list of comments
        level: Current nesting level (for indentation)
        max_depth: Maximum depth to traverse (None for unlimited)
    
    Returns:
        List of all comments with their nesting information
    """
    all_comments = []
    indent = "  " * level
    
    # Handle both single comment and list of comments
    if hasattr(comments, 'list'):
        comment_list = comments.list()
    else:
        comment_list = comments if isinstance(comments, list) else [comments]
    
    for comment in comment_list:
        # Skip MoreComments objects
        if hasattr(comment, 'body') and comment.body not in ['[deleted]', '[removed]']:
            comment_info = {
                'body': comment.body,
                'author': str(comment.author) if comment.author else "[deleted]",
                'score': comment.score,
                'level': level,
                'created_utc': comment.created_utc,
                'comment_id': comment.id
            }
            all_comments.append(comment_info)
            
            # Print with proper indentation
            print(f"{indent}Level {level}: {comment.body[:100]}{'...' if len(comment.body) > 100 else ''}")
            
            # Recursively process replies if they exist and we haven't hit max depth
            if hasattr(comment, 'replies') and comment.replies and (max_depth is None or level < max_depth):
                replies = iterate_comment_forest(comment.replies, level + 1, max_depth)
                all_comments.extend(replies)
    
    return all_comments

def print_all_comments_simple(comments, level=0):
    """
    Simple recursive function to print all comments with indentation
    """
    indent = "  " * level
    
    for comment in comments:
        if hasattr(comment, 'body') and comment.body not in ['[deleted]', '[removed]']:
            print(f"{indent}Level {level}: {comment.body}")
            
            # Process replies recursively
            if hasattr(comment, 'replies') and comment.replies:
                print_all_comments_simple(comment.replies, level + 1)

def get_all_comments_as_list(comments, level=0):
    """
    Get all comments as a flat list with level information
    """
    all_comments = []
    
    for comment in comments:
        if hasattr(comment, 'body') and comment.body not in ['[deleted]', '[removed]']:
            all_comments.append({
                'body': comment.body,
                'level': level,
                'author': str(comment.author) if comment.author else "[deleted]",
                'score': comment.score
            })
            
            # Recursively get replies
            if hasattr(comment, 'replies') and comment.replies:
                replies = get_all_comments_as_list(comment.replies, level + 1)
                all_comments.extend(replies)
    
    return all_comments

# Updated version of your code that iterates through the entire comment forest
def print_posts_with_full_comments(posts, max_depth=None):
    """
    Print posts with their entire comment forest
    """
    for i, n in enumerate(posts, 1):
        print(f"\n{'='*60}")
        print(f"POST {i}: {n.title}")
        print(f"Body: {n.selftext[:200]}{'...' if len(n.selftext) > 200 else ''}")
        print(f"Score: {n.score}, Total Comments: {n.num_comments}")
        print(f"{'='*60}")
        
        # Replace MoreComments to get all comments
        n.comments.replace_more(limit=0)
        
        # Print all comments in the forest
        print("COMMENT FOREST:")
        all_comments = iterate_comment_forest(n.comments, max_depth=max_depth)
        
        print(f"\nTotal comments processed: {len(all_comments)}")
        print("-" * 60)

print("Comment forest iteration functions loaded!")
print("Use print_posts_with_full_comments(news) to see the entire comment tree")
print("Use iterate_comment_forest(n.comments) to get all comments from a single post")


In [ ]:
# Replace your current code with this to iterate through the entire comment forest

def recursive_comment_print(comments, level=0):
    """Recursively print all comments with proper indentation"""
    indent = "  " * level
    
    for comment in comments:
        if hasattr(comment, 'body') and comment.body not in ['[deleted]', '[removed]']:
            print(f"{indent}Level {level}: {comment.body}")
            
            # Recursively print replies
            if hasattr(comment, 'replies') and comment.replies:
                recursive_comment_print(comment.replies, level + 1)

# Your updated code that iterates through the entire comment forest:
for n in news:
    print(f"Title: {n.title}")
    print(f"Body: {n.selftext}")
    print("Comments (Full Forest):")
    
    # Replace MoreComments objects with actual comments
    n.comments.replace_more(limit=0)
    
    # Recursively print all comments in the forest
    recursive_comment_print(n.comments)
    print("-" * 50)


In [ ]:
def build_comment_tree(comments, level=0, max_depth=None):
    """
    Build a hierarchical comment tree structure
    
    Args:
        comments: Reddit comment object or list of comments
        level: Current nesting level
        max_depth: Maximum depth to traverse (None for unlimited)
    
    Returns:
        List of comment dictionaries with nested replies
    """
    comment_tree = []
    
    # Handle both single comment and list of comments
    if hasattr(comments, 'list'):
        comment_list = comments.list()
    else:
        comment_list = comments if isinstance(comments, list) else [comments]
    
    for comment in comment_list:
        # Skip MoreComments objects and deleted/removed comments
        if hasattr(comment, 'body') and comment.body not in ['[deleted]', '[removed]']:
            comment_dict = {
                "body": comment.body,
                "author": str(comment.author) if comment.author else "[deleted]",
                "score": comment.score,
                "created_utc": comment.created_utc,
                "comment_id": comment.id,
                "level": level
            }
            
            # Recursively build replies if they exist and we haven't hit max depth
            if hasattr(comment, 'replies') and comment.replies and (max_depth is None or level < max_depth):
                replies = build_comment_tree(comment.replies, level + 1, max_depth)
                if replies:  # Only add replies if there are any
                    comment_dict["replies"] = replies
            
            comment_tree.append(comment_dict)
    
    return comment_tree

def get_posts_with_comment_forest(posts, max_depth=None):
    """
    Get posts with their complete comment forest in hierarchical format
    
    Args:
        posts: List of Reddit submission objects
        max_depth: Maximum comment depth to traverse
    
    Returns:
        List of dictionaries with post data and hierarchical comment forest
    """
    result = []
    
    for post in posts:
        # Build post data
        post_data = {
            "title": post.title,
            "body": post.selftext if post.selftext else "",
            "url": post.url,
            "score": post.score,
            "num_comments": post.num_comments,
            "created_utc": post.created_utc,
            "author": str(post.author) if post.author else "[deleted]",
            "subreddit": str(post.subreddit),
            "post_id": post.id
        }
        
        # Build comment forest
        try:
            # Replace MoreComments to get all comments
            post.comments.replace_more(limit=0)
            
            # Build hierarchical comment tree
            comment_forest = build_comment_tree(post.comments, max_depth=max_depth)
            
            # Add comment forest to post data
            if comment_forest:
                post_data["replies"] = comment_forest
            
        except Exception as e:
            print(f"Error building comment forest for post '{post.title}': {e}")
            post_data["replies"] = []
        
        result.append(post_data)
    
    return result

def get_single_post_with_comment_forest(post, max_depth=None):
    """
    Get a single post with its complete comment forest
    
    Args:
        post: Reddit submission object
        max_depth: Maximum comment depth to traverse
    
    Returns:
        Dictionary with post data and hierarchical comment forest
    """
    # Build post data
    post_data = {
        "title": post.title,
        "body": post.selftext if post.selftext else "",
        "url": post.url,
        "score": post.score,
        "num_comments": post.num_comments,
        "created_utc": post.created_utc,
        "author": str(post.author) if post.author else "[deleted]",
        "subreddit": str(post.subreddit),
        "post_id": post.id
    }
    
    # Build comment forest
    try:
        # Replace MoreComments to get all comments
        post.comments.replace_more(limit=0)
        
        # Build hierarchical comment tree
        comment_forest = build_comment_tree(post.comments, max_depth=max_depth)
        
        # Add comment forest to post data
        if comment_forest:
            post_data["replies"] = comment_forest
        else:
            post_data["replies"] = []
        
    except Exception as e:
        print(f"Error building comment forest for post '{post.title}': {e}")
        post_data["replies"] = []
    
    return post_data

# Example usage functions
def print_comment_tree(comment_dict, level=0):
    """
    Pretty print the comment tree structure
    """
    indent = "  " * level
    
    if isinstance(comment_dict, list):
        for comment in comment_dict:
            print_comment_tree(comment, level)
    else:
        print(f"{indent}Level {level}: {comment_dict['body'][:100]}{'...' if len(comment_dict['body']) > 100 else ''}")
        print(f"{indent}  Author: {comment_dict['author']}, Score: {comment_dict['score']}")
        
        if 'replies' in comment_dict and comment_dict['replies']:
            print(f"{indent}  Replies:")
            print_comment_tree(comment_dict['replies'], level + 1)

def analyze_comment_forest(post_data):
    """
    Analyze the comment forest structure
    """
    def count_comments(comments, level=0):
        total = len(comments)
        max_level = level
        
        for comment in comments:
            if 'replies' in comment and comment['replies']:
                sub_total, sub_max_level = count_comments(comment['replies'], level + 1)
                total += sub_total
                max_level = max(max_level, sub_max_level)
        
        return total, max_level
    
    if 'replies' in post_data and post_data['replies']:
        total_comments, max_depth = count_comments(post_data['replies'])
        print(f"Comment Forest Analysis:")
        print(f"  Total comments: {total_comments}")
        print(f"  Maximum depth: {max_depth}")
        print(f"  Top-level comments: {len(post_data['replies'])}")
    else:
        print("No comments found")

print("Enhanced comment forest functions loaded!")
print("Use get_posts_with_comment_forest(news) to get hierarchical comment structure")
print("Use get_single_post_with_comment_forest(post) for a single post")
print("Use print_comment_tree(post_data['replies']) to visualize the structure")


In [ ]:
import praw

reddit = praw.Reddit(client_id='qkvvzxj-rLgMYUWf9FeXvg', client_secret='6clslIMBWxqCs8hG4GYpmwj09u807g', user_agent='ZZZZZZ')

In [ ]:
subreddit = reddit.subreddit('wallstreetbets')


In [14]:
# Example usage of the improved comment forest function
news = subreddit.new(limit=10)

# Get all posts with their complete comment forests
posts_with_forests = get_posts_with_comment_forest(news, max_depth=None)


import json
jsonstr = json.dumps(posts_with_forests, indent=2, ensure_ascii=False)
print(jsonstr)



[
  {
    "title": "Too many profit posts, here’s a juicy loss. A very tiny minority actually wins in this game.",
    "body": " ",
    "url": "https://www.reddit.com/gallery/1nyyhex",
    "score": 147,
    "num_comments": 54,
    "created_utc": 1759694600.0,
    "author": "Short-Dependent7676",
    "subreddit": "wallstreetbets",
    "post_id": "1nyyhex",
    "replies": [
      {
        "body": "\n**User Report**| | | |\n:--|:--|:--|:--\n**Total Submissions** | 5 | **First Seen In WSB** | 4 months ago\n**Total Comments** | 518 | **Previous Best DD** | \n**Account Age** | 1 year | | \n\n[**Join WSB Discord**](https://discord.gg/wsbverse) | [**⚔**](https://www.reddit.com/r/KarmaCave/)",
        "author": "VisualMod",
        "score": 1,
        "created_utc": 1759694610.0,
        "comment_id": "nhy7fbv",
        "level": 0
      },
      {
        "body": "Jeez my dude… How tf you lost money with calls in this market?! Stay strong man 🙏",
        "author": "resunzz",
        "score": 6

In [ ]:

# Print the first post's structure
if posts_with_forests:
    first_post = posts_with_forests[0]
    print(f"Post: {first_post['title']}")
    print(f"Body: {first_post['body'][:100]}...")
    print(f"Total comments: {first_post['num_comments']}")
    
    # Analyze the comment forest
    analyze_comment_forest(first_post)
    
    # Print the comment tree structure
    print("\nComment Tree Structure:")
    print_comment_tree(first_post['replies'])
    
    # Convert to JSON to see the full structure
    import json
    print(f"\nJSON Structure (first 1000 chars):")
    json_str = json.dumps(first_post, indent=2, ensure_ascii=False)
    print(json_str[:1000] + "..." if len(json_str) > 1000 else json_str)

# Alternative: Get just one post with its comment forest
# single_post = get_single_post_with_comment_forest(news[0])
# print(json.dumps(single_post, indent=2))


In [ ]:
from praw.reddit import Submission


for n in news:
    print(f"Title: {n.title}")
    print(f"Body: {n.selftext}")
    print("Comments:")
    
    # This is the key fix - replace MoreComments objects with actual comments
    n.comments.replace_more(limit=0)
    
    # Now iterate through the actual comments
    for c in n.comments.list():
        if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']:
            print(f'    {c.body}')
    print("-" * 50)

In [ ]:
print_posts_with_full_comments(news)